# MiniCPM5-1B address repair: brief GRPO run (v4, second attempt)

One short GRPO run on top of the frozen v3 SFT adapter. Group of 4 answers
per record, reward = structure + per-field repair \u2212 damage \u2212 fills
\u2212 additions + honest review (see `src/addr_repair/rewards.py`).
Checkpoint picked on val repair F1 with damage and fill counts as guards,
never on recall alone.

Why v4-second-attempt: the first attempt failed before training (GRPO code
never left the Mac, Drive paths differed, merge double-applied v3, GGUF
corrupt). This version takes the code as a Drive pack, gates training on a
passing transfer check, and sanity-checks the final GGUF.

**Stop rules (plan step 6):** one brief run only. Stop sooner if damage or
empty fills rise above v3 (damage `0.0237`, fills `1`), even if F1 rises.
A higher F1 built on guessing does not count as a win.

**Bring back to the Mac:** `checkpoints-grpo-v4/selection.json` (winner +
scores), `training_record.json`, and the sha256 of
`sft-Q4_K_M-grpo-v4.gguf`. Frozen 4-way eval (step 7) runs locally.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/TMFNK/local-slm-de-address-repair.git
%cd local-slm-de-address-repair
!pip install -q uv

In [ ]:
# Install ALL deps incl. dev extras (pytest lives there; plain uv run skips it).
!uv sync --extra dev
!uv run python -c "import torch, trl, transformers; print(torch.__version__, trl.__version__, transformers.__version__, torch.cuda.is_available())"


Code transfer: the GRPO files are not on GitHub yet, so upload
`grpo-v4-pack.zip` to Drive root (`/content/drive/MyDrive/`) first.
The next two cells unpack it over the clone and prove the transfer with
the committed tests. Do not train if the gate cell fails.

In [ ]:
!unzip -o /content/drive/MyDrive/grpo-v4-pack.zip -d /tmp/grpo-pack
!cp /tmp/grpo-pack/scripts/train_grpo.py scripts/
!cp /tmp/grpo-pack/src/addr_repair/rewards.py src/addr_repair/
!cp /tmp/grpo-pack/configs/train_grpo.yaml configs/
!cp /tmp/grpo-pack/tests/test_train_grpo.py tests/
!cp /tmp/grpo-pack/tests/test_rewards.py tests/
!ls -la scripts/train_grpo.py src/addr_repair/rewards.py configs/train_grpo.yaml

In [ ]:
# Transfer gate: new reward + GRPO tests must pass before any GPU work.
!uv run pytest -q tests/test_rewards.py tests/test_train_grpo.py

Data: copy the two slice CSVs from Drive (the T4 path). Hashes must match
`configs/data.yaml`:
- dirty `2ef8ab1424c8c357af1aeaecef12b376cb7f20616437035f2753d0b2f2ac86ef`
- clean `78c852e3a5680a7460732b67ab87d82484a718297190fa8852af73a80817a192`
Stop here if they do not match: the first attempt saw mismatched row
counts on these Drive copies.

In [ ]:
!mkdir -p data/raw
!cp "/content/drive/MyDrive/colab-data/dirty.csv" data/raw/dirty.csv
!cp "/content/drive/MyDrive/colab-data/clean.csv" data/raw/clean.csv

In [ ]:
!uv run python -c "import csv; [print(f, sum(1 for _ in open(f)) - 1, 'rows') for f in ('data/raw/dirty.csv', 'data/raw/clean.csv')]"
!sha256sum data/raw/dirty.csv data/raw/clean.csv

In [ ]:
# Drive copies, no zip on disk (T4 run):
!uv run python scripts/prepare_data.py --config configs/data.yaml --skip-archive --force

Stage the frozen v3 adapter. The winner name comes from the committed
`evals/sft-v3/selection.json` (v3 winner: `checkpoint-800`); its weights
live under the `-clean-gold-v3` Drive tree. Expected adapter sha256 starts
with `7103451b`; stop if it does not match.

In [ ]:
%%writefile /content/stage_v3_adapter.py
"""Copy the SFT winner checkpoint to the GRPO init-adapter path."""
import hashlib
import json
import shutil
from pathlib import Path

V3ROOT = Path("/content/drive/MyDrive/local-slm-de-address-repair-clean-gold-v3")
GROOT = Path("/content/drive/MyDrive/local-slm-de-address-repair")
winner = json.load(open("/content/local-slm-de-address-repair/evals/sft-v3/selection.json"))["winner"]
src = V3ROOT / "checkpoints" / winner
assert (src / "adapter_model.safetensors").is_file(), f"no adapter weights in {src}"
dst = GROOT / "adapters" / "clean-gold-v3"
print("v3 winner:", winner)
if dst.is_dir():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
digest = hashlib.sha256()
with (dst / "adapter_model.safetensors").open("rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        digest.update(chunk)
print("staged:", dst)
print("adapter sha256:", digest.hexdigest())
print("expected   : 7103451b9cc916920c59e5d68e8c3ada852b294be5eab55a0d1a9abdc77ec712")

In [ ]:
!uv run python /content/stage_v3_adapter.py

Dry run first: no GPU, no writes. Must print the 1,500-row plan and the
reward weights with no warnings before training starts.

In [ ]:
%cd /content/local-slm-de-address-repair
!uv run python scripts/train_grpo.py --config configs/train_grpo.yaml --dry-run

In [ ]:
# Brief run: 150 steps, ~8 prompts/step, 4 completions each.
!uv run python scripts/train_grpo.py --config configs/train_grpo.yaml
# After a disconnect: redo clone, pack, data, prepare, stage cells, then:
# !uv run python scripts/train_grpo.py --config configs/train_grpo.yaml --resume
# (Resume with no steps left still runs post-hoc val scoring.)

In [ ]:
!cat /content/drive/MyDrive/local-slm-de-address-repair/checkpoints-grpo-v4/selection.json

In [ ]:
!uv run python -c "import json; r = json.load(open('/content/drive/MyDrive/local-slm-de-address-repair/checkpoints-grpo-v4/training_record.json')); print({k: r[k] for k in ('winner', 'winner_selection_score', 'train_seconds', 'val_scoring_seconds')}); print('reward:', r['reward_weights']); print(r['gpu'])"

Merge the winner for local eval. The GRPO adapter sits on top of the v3
adapter: merge v3 into base first, then the GRPO winner ONCE on top.
(The first attempt applied v3 twice and no GRPO adapter \u2014 that file
was corrupt and discarded.) Then export GGUF at llama.cpp
`b31b71f3a076bfc4278daad442203a9c51c6e676` (same rev as v3).
Filenames carry `-grpo-v4`; v3 artifacts are never overwritten. The last
cell refuses a bad GGUF: stop if it fails.

In [ ]:
%%writefile /content/merge_grpo.py
import json
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id, rev = "openbmb/MiniCPM5-1B", "87179e5c1f455ef22e6223592d2d61351b525bfc"
groot = "/content/drive/MyDrive/local-slm-de-address-repair"
v3_path = groot + "/adapters/clean-gold-v3"
grpo_root = groot + "/checkpoints-grpo-v4"
winner = json.load(open(grpo_root + "/selection.json"))["winner"]
grpo_path = f"{grpo_root}/{winner}"
assert Path(v3_path, "adapter_model.safetensors").is_file(), v3_path
assert Path(grpo_path, "adapter_model.safetensors").is_file(), grpo_path
assert v3_path != grpo_path, "GRPO adapter must differ from the v3 adapter"
out = groot + "/merged-grpo-v4"
tok = AutoTokenizer.from_pretrained(base_id, revision=rev)
base = AutoModelForCausalLM.from_pretrained(
    base_id, revision=rev, torch_dtype="auto", device_map="cpu"
)
with_v3 = PeftModel.from_pretrained(base, v3_path).merge_and_unload()
full = PeftModel.from_pretrained(with_v3, grpo_path).merge_and_unload()
full.save_pretrained(out)
tok.save_pretrained(out)
assert Path(out, "model.safetensors").is_file(), "merge saved no weights"
print("merged v3 +", winner, "->", out)

In [ ]:
!uv run python /content/merge_grpo.py

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp && cd /content/llama.cpp && git checkout b31b71f3a076bfc4278daad442203a9c51c6e676
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j2 --target llama-quantize

In [ ]:
!uv run --with gguf --with sentencepiece python /content/llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/local-slm-de-address-repair/merged-grpo-v4 --outfile /content/drive/MyDrive/local-slm-de-address-repair/sft-grpo-v4-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize /content/drive/MyDrive/local-slm-de-address-repair/sft-grpo-v4-f16.gguf /content/drive/MyDrive/local-slm-de-address-repair/sft-Q4_K_M-grpo-v4.gguf Q4_K_M
!uv run python -c "
from pathlib import Path
import hashlib
p = Path('/content/drive/MyDrive/local-slm-de-address-repair/sft-Q4_K_M-grpo-v4.gguf')
head = p.read_bytes()[:4]
assert head == b'GGUF', f'bad GGUF magic: {head!r} (re-download, do not use this file)'
assert p.stat().st_size > 600_000_000, f'suspicious size: {p.stat().st_size} (v3 is 688,065,792)'
h = hashlib.sha256()
with p.open('rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
print('GGUF OK, bytes:', p.stat().st_size)
print('sha256:', h.hexdigest())
"